# ChaseEscapeEnv PPO Training (Colab)

This notebook trains a PPO agent for the 2D pursuit-evasion environment **only in Colab**.

- Uses GPU (e.g. T4) if enabled in Colab
- Keeps your local machine for running the Pygame sandbox only
- Saves the trained model to `runs/ppo_chase_escape/ppo_chase_escape_final.zip`

You can later download that model and play it locally or in Colab.

In [ ]:
# Clone the pursuit_arena repository in Colab and move into the project folder.
# This uses your GitHub repo: https://github.com/GraslyDias/pursuit_arena.git

%cd /content
!git clone https://github.com/GraslyDias/pursuit_arena.git
%cd /content/pursuit_arena

In [ ]:
# Install dependencies in Colab only (NOT on your local machine).

!pip install pygame gymnasium stable-baselines3[extra] numpy

In [ ]:
# Optional: quick environment check
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv
from stable_baselines3.common.env_checker import check_env

env = ChaseEscapeEnv(render_mode=None)
check_env(env, warn=True)
env.close()

In [ ]:
# Train PPO on ChaseEscapeEnv
# Model path: runs/ppo_chase_escape/ppo_chase_escape_final.zip (upload there in Colab to re-train from current zip).
# Optional: upload training_map.json to train on a fixed map.
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv, load_training_map

log_dir = Path("runs/ppo_chase_escape")
log_dir.mkdir(parents=True, exist_ok=True)

def make_env():
    env = ChaseEscapeEnv(render_mode=None)
    # Use saved map for training if present (from play_model "Save map")
    map_path = Path("training_map.json")
    if map_path.exists():
        env.training_map_options = load_training_map(map_path)
        print("Training on saved map: training_map.json")
    env = Monitor(env, str(log_dir / "monitor.csv"))
    return env

vec_env = DummyVecEnv([make_env])

# Re-train from current zip to improve (continue training). If no zip exists, train from scratch.
model_path = log_dir / "ppo_chase_escape_final.zip"
if model_path.exists():
    model = PPO.load(str(model_path), env=vec_env)
    print("Loaded existing model for continued training.")
else:
    model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb"))
    print("Training from scratch.")
model.learn(total_timesteps=300_000)
model.save(str(model_path))
print("Saved improved model to", model_path)
vec_env.close()

In [ ]:
# Evaluate the trained model in Colab (no rendering window)
from stable_baselines3 import PPO
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv

env = ChaseEscapeEnv(render_mode=None)
model = PPO.load("runs/ppo_chase_escape/ppo_chase_escape_final.zip", env=env)

episodes = 5
for ep in range(episodes):
    obs, info = env.reset()
    done = False
    truncated = False
    ep_reward = 0.0
    while not (done or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        ep_reward += float(reward)
    print(f"Episode {ep+1}: reward={ep_reward:.2f}, info={info}")

env.close()

In [ ]:
# Download the trained model to your local machine (from Colab)
from google.colab import files

files.download("runs/ppo_chase_escape/ppo_chase_escape_final.zip")